<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/19_neural_networks/19_1_Neural_Networks/19_1_3_Multiclass_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Neural Networks: Part 3
## Multiclass Classification

---

## What This Notebook Is About

The last two notebooks trained binary classifiers — one output neuron, sigmoid activation, BCELoss. Most real classification problems have more than two classes. This notebook builds a multiclass neural network and applies it to a dataset you have already worked with.

In unit 18_5, you classified the **CTG fetal health** dataset using XGBoost: three classes (Normal, Suspect, Pathological), 35 features, 2,126 samples. You learned about macro F1, per-class recall, and class imbalance. This notebook asks: can a neural network do the same job, using the same evaluation framework?

The answer will introduce three new PyTorch concepts:
1. **Softmax output layer** — replaces sigmoid; outputs K probabilities that sum to 1 (you already know softmax from units 18_5 and 18_6)
2. **`CrossEntropyLoss`** — the multiclass equivalent of BCELoss; combines softmax and log-loss in one numerically stable operation
3. **Class weights in the loss** — the same idea as `compute_sample_weight('balanced')` from unit 18_5, now applied to the loss function

**What you will learn:**
1. How the output layer changes when going from binary to multiclass
2. Why `CrossEntropyLoss` takes raw logits, not softmax probabilities
3. How to weight the loss by inverse class frequency to handle imbalance
4. How to get probabilities and class labels from a PyTorch multiclass model
5. How to evaluate with sklearn's `classification_report` and compare to XGBoost

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
from sklearn.utils import compute_class_weight

import seaborn as sns
sns.set_theme(style='whitegrid')
torch.manual_seed(42)

# Load the CTG fetal health dataset (same as unit 18_5_3)
data_ctg = fetch_openml(name='cardiotocography', version=2, as_frame=True, parser='auto')
df = data_ctg.frame.copy()

class_names = ['Normal', 'Suspect', 'Pathological']
label_map   = {'1': 0, '2': 1, '3': 2}   # integer codes from OpenML
df['Class'] = df['Class'].astype(str).map(label_map)
df = df.dropna().reset_index(drop=True)

X_raw = df.drop(columns=['Class']).to_numpy(dtype=np.float32)
y_raw = df['Class'].to_numpy(dtype=np.int64)

print(f'Dataset: {X_raw.shape[0]} samples, {X_raw.shape[1]} features')
print('Class distribution:')
for i, name in enumerate(class_names):
    count = (y_raw == i).sum()
    print(f'  {name:<14}  {count:>4}  ({count/len(y_raw):.1%})')

In [ ]:
# Three-way split: 60% train / 20% val / 20% test  (same pattern as 19_1_2)
X_tmp, X_test_np, y_tmp, y_test_np = train_test_split(
    X_raw, y_raw, test_size=0.20, stratify=y_raw, random_state=42
)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=42
)

scaler     = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_val_np   = scaler.transform(X_val_np)
X_test_np  = scaler.transform(X_test_np)

X_train = torch.tensor(X_train_np)
X_val   = torch.tensor(X_val_np)
X_test  = torch.tensor(X_test_np)
y_train = torch.tensor(y_train_np)   # shape (n,) — long integers, NOT floats
y_val   = torch.tensor(y_val_np)
y_test  = torch.tensor(y_test_np)

print(f'Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}')
print(f'y dtype: {y_train.dtype}  (CrossEntropyLoss requires torch.long)')

BATCH_SIZE   = 64
train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)

---

## Section 1: From Sigmoid to Softmax

For binary classification, the output layer had one neuron and a sigmoid activation — it produced one number in (0, 1) representing P(class=1).

For K classes, the output layer has **K neurons** — one per class. Each neuron produces a raw score (called a **logit**). To turn K logits into K probabilities that sum to 1, we apply **softmax**:

    softmax(z₁, z₂, ..., z_K)ᵢ = exp(zᵢ) / Σⱼ exp(zⱼ)

You have seen this function before:
- Unit 18_5_1: `objective='multi:softprob'` in XGBoost uses softmax internally
- Unit 18_5_1: `predict_proba()` returns the same N×K probability matrix
- Unit 18_5_1 summary table: "softmax converts K raw scores to K probabilities that sum to 1.0"

The network output will be the same format as XGBoost's `predict_proba()` — which you already know how to read.

### The `CrossEntropyLoss` convention

Here is the one convention that trips up almost every new user: **`nn.CrossEntropyLoss` expects raw logits, not softmax probabilities**. It applies softmax internally, combined with the log-loss computation, in one numerically stable step.

This means the model's `forward` method should NOT apply softmax. The final layer is just `nn.Linear(hidden, K)` — no activation. The sigmoid you used for binary classification is gone.

| Task | Output layer | Loss function | Target shape |
|---|---|---|---|
| Binary | `nn.Linear(h, 1)` + sigmoid | `nn.BCELoss()` | (n, 1) float |
| Multiclass | `nn.Linear(h, K)` — no activation | `nn.CrossEntropyLoss()` | (n,) long int |

In [ ]:
class CTGNet(nn.Module):
    def __init__(self, n_features=35, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes)   # K outputs — NO sigmoid or softmax here
        )

    def forward(self, x):
        return self.net(x)   # returns raw logits; CrossEntropyLoss applies softmax


# Verify the output shape
torch.manual_seed(0)
demo_model = CTGNet()
demo_input = torch.randn(4, 35)   # 4 samples, 35 features
demo_out   = demo_model(demo_input)

print(f'Input shape : {demo_input.shape}')
print(f'Output shape: {demo_out.shape}  (4 samples × 3 class logits)')
print(f'Raw logits  : {demo_out.detach().round(decimals=3)}')
print()
print('To get probabilities from logits:')
probs = torch.softmax(demo_out, dim=1)
print(f'After softmax: {probs.detach().round(decimals=3)}')
print(f'Row sums     : {probs.sum(dim=1).detach()}')

---

## Section 2: Handling Class Imbalance

Normal makes up 78% of this dataset. A model that always predicts "Normal" would be right 78% of the time and catch zero Pathological cases. You tackled this in unit 18_5_3 using `compute_sample_weight('balanced')` passed to `model.fit()`.

In PyTorch, the equivalent is the `weight` argument of `nn.CrossEntropyLoss`. It takes a tensor of K class weights — the same values that `compute_class_weight('balanced')` would produce. During training, each sample's loss contribution is multiplied by the weight of its true class, so rare classes receive proportionally more gradient signal.

The math is the same as in 18_5_3; only the API is different.

In [ ]:
# Compute balanced class weights from training labels
class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(3),
    y=y_train_np
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32)

print('Class weights (inverse frequency):')
for name, w in zip(class_names, class_weights):
    count = (y_train_np == class_names.index(name)).sum()
    print(f'  {name:<14}  count={count:>4}  weight={w:.3f}')
print()
print('Normal gets low weight (most common); Pathological gets high weight (rarest).')
print('This matches the sample_weight logic from unit 18_5_3.')

---

## Section 3: Training the Multiclass Network

The training loop is identical to notebook 19_1_2 — the only change is the loss function.

We pass `weight=class_weights` to `CrossEntropyLoss` and train with Adam. After each epoch, we track both train and validation **macro F1** rather than loss — macro F1 is the headline metric for imbalanced multiclass problems, as established in unit 18_5_3.

In [ ]:
def get_preds(model, X):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        return logits.argmax(dim=1).numpy()


torch.manual_seed(42)
model     = CTGNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)

n_epochs        = 150
train_f1_hist   = []
val_f1_hist     = []
train_loss_hist = []

for epoch in range(1, n_epochs + 1):
    model.train()
    epoch_loss = 0.0
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_loss_hist.append(epoch_loss / len(train_loader))

    # Track macro F1 on both splits
    train_f1 = f1_score(y_train_np, get_preds(model, X_train), average='macro')
    val_f1   = f1_score(y_val_np,   get_preds(model, X_val),   average='macro')
    train_f1_hist.append(train_f1)
    val_f1_hist.append(val_f1)
    model.train()

    if epoch % 30 == 0:
        print(f'  epoch {epoch:>3}  loss: {train_loss_hist[-1]:.4f}  '
              f'train macro F1: {train_f1:.3f}  val macro F1: {val_f1:.3f}')

# Plot macro F1 curves
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_f1_hist, label='train macro F1', color='steelblue', lw=2)
ax.plot(val_f1_hist,   label='val macro F1',   color='#C0392B',   lw=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Macro F1')
ax.set_title('Macro F1 over Training  (CTGNet with balanced class weights)')
ax.legend()
plt.tight_layout()
plt.show()

---

## Section 4: The Probability Output

In unit 18_5_1 you learned that `predict_proba()` from XGBoost returns an N×K matrix — one row per sample, one column per class, each row summing to 1.0. This is exactly what `torch.softmax(model(X), dim=1)` gives you.

The predicted class is `argmax` across the K columns — whichever class has the highest probability.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_val[:6])
    probs  = torch.softmax(logits, dim=1)
    preds  = logits.argmax(dim=1)

print('First 6 validation samples — probability output (same format as XGBoost predict_proba):')
print()
header = f'  {"Actual":<14}  {"Pred":<14}  '
header += '  '.join(f'P({n})' for n in class_names)
print(header)
print('  ' + '-' * (len(header) - 2))
for i in range(6):
    actual = class_names[y_val[i].item()]
    pred   = class_names[preds[i].item()]
    p_str  = '  '.join(f'{probs[i, j].item():.3f}   ' for j in range(3))
    match  = '✓' if actual == pred else '✗'
    print(f'  {actual:<14}  {pred:<14}  {p_str}  {match}')
print()
print('Row sums:', probs.sum(dim=1).round(decimals=6).tolist())

---

## Section 5: Evaluation

Same evaluation framework as unit 18_5: `classification_report` with macro F1 as the headline metric, and a confusion matrix to see which class pairs the model confuses.

In [ ]:
# Train a final model on train+val combined
X_trainval = torch.cat([X_train, X_val])
y_trainval = torch.cat([y_train, y_val])

tv_loader = DataLoader(TensorDataset(X_trainval, y_trainval),
                       batch_size=BATCH_SIZE, shuffle=True)

# Recompute class weights for the combined set
cw_np  = compute_class_weight('balanced', classes=np.arange(3),
                               y=torch.cat([y_train, y_val]).numpy())
cw     = torch.tensor(cw_np, dtype=torch.float32)

torch.manual_seed(42)
final  = CTGNet()
fopt   = torch.optim.Adam(final.parameters(), lr=1e-3, weight_decay=1e-4)
fcrit  = nn.CrossEntropyLoss(weight=cw)

for epoch in range(150):
    final.train()
    for X_b, y_b in tv_loader:
        fopt.zero_grad()
        fcrit(final(X_b), y_b).backward()
        fopt.step()

final.eval()
with torch.no_grad():
    test_preds = final(X_test).argmax(dim=1).numpy()

print('Classification report — CTGNet on held-out test set:')
print()
print(classification_report(y_test_np, test_preds, target_names=class_names, digits=3))

macro_f1 = f1_score(y_test_np, test_preds, average='macro')
print(f'Test macro F1 : {macro_f1:.3f}')
print()
print('For comparison, XGBoost (balanced, no tuning) from unit 18_5_3 achieved ~0.85-0.90')
print('macro F1 on this dataset via 5-fold CV.')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_np, test_preds,
    display_labels=class_names,
    cmap='Blues',
    ax=ax
)
ax.set_title('CTGNet Confusion Matrix — Test Set')
ax.grid(False)
plt.tight_layout()
plt.show()

---

## Putting It All Together

| Concept | Binary (19_1_2) | Multiclass (this notebook) |
|---|---|---|
| Output layer | `nn.Linear(h, 1)` + sigmoid | `nn.Linear(h, K)` — no activation |
| Loss function | `nn.BCELoss()` | `nn.CrossEntropyLoss()` |
| Target tensor | float, shape (n, 1) | long int, shape (n,) |
| Probabilities | `model(X)` directly | `torch.softmax(model(X), dim=1)` |
| Predicted class | `(model(X) > 0.5).long()` | `model(X).argmax(dim=1)` |
| Class imbalance | `weight_decay` (regularization) | `CrossEntropyLoss(weight=class_weights)` |
| Headline metric | Accuracy or AUC | Macro F1 (per-class recall for high-stakes classes) |

**The `CrossEntropyLoss` convention:** The loss function applies softmax internally. The model's `forward` method returns raw logits. Applying softmax in the model AND using `CrossEntropyLoss` would double-apply softmax and produce wrong gradients — a common and silent error.

**Connecting back to unit 18_5:** The probability output (`torch.softmax(model(X), dim=1)`) is the same N×K matrix as `predict_proba()` in XGBoost. The class weights (`CrossEntropyLoss(weight=...)`) are the PyTorch equivalent of `compute_sample_weight('balanced')`. The evaluation tools — `classification_report`, confusion matrix, macro F1 — are identical. Only the model changed.

**What is coming next:** Notebook 19_1_9 is the exercise: you will apply the full pipeline independently to the wine quality dataset from unit 18_5_9, and compare your neural network to the XGBoost model you built there.